# Modeling

This notebook trains and evaluates models that predict point outcome using only pre-serve information.

The goal is not just accuracy, but interpretability: understanding which serve features and match contexts are most associated with winning points.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

df = pd.read_csv("../data/processed/table_tennis_serves_features.csv")
df.head()

In [ ]:
model_features = ["serve_type","spin_type","spin_intensity","serve_length","placement_zone","toss_height","contact_point","game_number","server_score","receiver_score","game_state","opponent_skill_level","opponent_style","side","intended_setup","score_margin","total_points_played_in_game","is_tied","is_trailing","is_leading","is_late_game","is_deuce_or_later","is_game_point_for_server","is_game_point_against_server","serve_spin_combo","serve_length_spin_combo","serve_placement_combo","full_serve_combo","is_heavy_spin","is_low_spin","combo_attempts","combo_win_rate","combo_reliability"]
X = df[model_features]
y = df["point_won"]
groups = df["match_id"]

The model excludes post-serve variables such as return quality, rally length, and point-end type to avoid data leakage.

In [ ]:
baseline_accuracy = y.value_counts(normalize=True).max()
baseline_accuracy

In [ ]:
print(f"Baseline accuracy (always predict majority class): {baseline_accuracy:.3f}")
print("Class distribution:")
print(y.value_counts())

In [ ]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()
preprocessor = ColumnTransformer(transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),("num", StandardScaler(), numeric_features)])

In [ ]:
cv = GroupKFold(n_splits=5)

GroupKFold is used so that points from the same match do not appear in both the training and validation sets. This creates a more realistic estimate of model performance.

In [ ]:
lasso_model = Pipeline(steps=[("preprocessor", preprocessor),("classifier", LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000, class_weight="balanced"))])
lasso_accuracy = cross_val_score(lasso_model, X, y, cv=cv, groups=groups, scoring="accuracy")
lasso_auc = cross_val_score(lasso_model, X, y, cv=cv, groups=groups, scoring="roc_auc")
print(f"LASSO Accuracy: {lasso_accuracy.mean():.3f} \u00b1 {lasso_accuracy.std():.3f}")
print(f"LASSO ROC-AUC: {lasso_auc.mean():.3f} \u00b1 {lasso_auc.std():.3f}")

In [ ]:
lasso_fold_results = pd.DataFrame({
    "fold": list(range(1, 6)),
    "accuracy": lasso_accuracy,
    "roc_auc": lasso_auc
})
print("LASSO per-fold scores:")
lasso_fold_results

In [ ]:
lasso_model.fit(X, y)
feature_names = lasso_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = lasso_model.named_steps["classifier"].coef_[0]
coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
important_features = coef_df[coef_df["coefficient"] != 0].sort_values("abs_coefficient", ascending=False)
important_features.head(20)

In [ ]:
top15_lasso = important_features.head(15).sort_values("abs_coefficient", ascending=True)
colors = ["green" if c > 0 else "red" for c in top15_lasso["coefficient"]]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top15_lasso["feature"], top15_lasso["coefficient"], color=colors)
ax.set_title("Top 15 LASSO Feature Coefficients")
ax.set_xlabel("Coefficient Value")
ax.set_ylabel("Feature")
ax.axvline(x=0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()

Positive coefficients are associated with higher predicted probability of winning the point. Negative coefficients are associated with lower predicted probability.

In [ ]:
rf_model = Pipeline(steps=[("preprocessor", preprocessor),("classifier", RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42, class_weight="balanced"))])
rf_accuracy = cross_val_score(rf_model, X, y, cv=cv, groups=groups, scoring="accuracy")
rf_auc = cross_val_score(rf_model, X, y, cv=cv, groups=groups, scoring="roc_auc")
print(f"Random Forest Accuracy: {rf_accuracy.mean():.3f} \u00b1 {rf_accuracy.std():.3f}")
print(f"Random Forest ROC-AUC: {rf_auc.mean():.3f} \u00b1 {rf_auc.std():.3f}")

In [ ]:
rf_fold_results = pd.DataFrame({
    "fold": list(range(1, 6)),
    "accuracy": rf_accuracy,
    "roc_auc": rf_auc
})
print("Random Forest per-fold scores:")
rf_fold_results

In [ ]:
rf_model.fit(X, y)
rf_feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()
rf_importances = rf_model.named_steps["classifier"].feature_importances_
rf_importance_df = pd.DataFrame({"feature": rf_feature_names, "importance": rf_importances})
rf_importance_df = rf_importance_df.sort_values("importance", ascending=False).head(20)
print("Top 20 Random Forest Feature Importances:")
rf_importance_df

In [ ]:
top15_rf = rf_importance_df.head(15).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top15_rf["feature"], top15_rf["importance"], color="steelblue")
ax.set_title("Top 15 Random Forest Feature Importances")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

## Gradient Boosting Classifier

Gradient Boosting is a strong benchmark that often outperforms Random Forest by building trees sequentially to correct prior errors.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42))
])
gb_accuracy = cross_val_score(gb_model, X, y, cv=cv, groups=groups, scoring="accuracy")
gb_auc = cross_val_score(gb_model, X, y, cv=cv, groups=groups, scoring="roc_auc")
print(f"Gradient Boosting Accuracy: {gb_accuracy.mean():.3f} \u00b1 {gb_accuracy.std():.3f}")
print(f"Gradient Boosting ROC-AUC:  {gb_auc.mean():.3f} \u00b1 {gb_auc.std():.3f}")

In [ ]:
gb_fold_results = pd.DataFrame({
    "fold": list(range(1, 6)),
    "accuracy": gb_accuracy,
    "roc_auc": gb_auc
})
print("Gradient Boosting per-fold scores:")
gb_fold_results

In [ ]:
model_results = pd.DataFrame({
    "model": ["Baseline", "LASSO Logistic Regression", "Random Forest", "Gradient Boosting"],
    "accuracy": [baseline_accuracy, lasso_accuracy.mean(), rf_accuracy.mean(), gb_accuracy.mean()],
    "accuracy_std": [0, lasso_accuracy.std(), rf_accuracy.std(), gb_accuracy.std()],
    "roc_auc": [np.nan, lasso_auc.mean(), rf_auc.mean(), gb_auc.mean()],
    "roc_auc_std": [np.nan, lasso_auc.std(), rf_auc.std(), gb_auc.std()]
})
model_results

In [ ]:
models = model_results["model"].tolist()
accuracy_vals = model_results["accuracy"].tolist()
roc_auc_vals = model_results["roc_auc"].fillna(0).tolist()

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width / 2, accuracy_vals, width, label="Accuracy", color="steelblue")
bars2 = ax.bar(x + width / 2, roc_auc_vals, width, label="ROC-AUC", color="darkorange")

ax.set_title("Model Comparison: Accuracy and ROC-AUC")
ax.set_ylabel("Score")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.1)
ax.legend()

for bar in bars1:
    ax.annotate(f"{bar.get_height():.3f}", xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f"{height:.3f}", xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## Model Calibration

A well-calibrated model has predicted probabilities that match empirical win rates — e.g., serves predicted at 60% should win 60% of the time. We use `cross_val_predict` to get out-of-fold probabilities and compare them to actual outcomes.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.calibration import calibration_curve

# Get out-of-fold predicted probabilities for RF and GB
rf_probs = cross_val_predict(rf_model, X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
gb_probs = cross_val_predict(gb_model, X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]

fig, ax = plt.subplots(figsize=(6, 5))

for probs, label, color in [(rf_probs, "Random Forest", "steelblue"), (gb_probs, "Gradient Boosting", "darkorange")]:
    fraction_of_positives, mean_predicted_value = calibration_curve(y, probs, n_bins=8)
    ax.plot(mean_predicted_value, fraction_of_positives, marker="o", label=label, color=color)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives (Actual Win Rate)")
ax.set_title("Calibration Curve")
ax.legend()
plt.tight_layout()
plt.show()

## Confusion Matrix

The confusion matrix shows how many points the model correctly classified, broken down by true vs predicted outcome.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best_probs = rf_probs  # use RF as the primary model
best_preds = (best_probs >= 0.5).astype(int)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y, best_preds, display_labels=["Lost", "Won"],
                                         colorbar=False, ax=ax, cmap="Blues")
ax.set_title("Random Forest Confusion Matrix\n(cross-validated, threshold = 0.50)")
plt.tight_layout()
plt.show()

from sklearn.metrics import classification_report
print(classification_report(y, best_preds, target_names=["Lost", "Won"]))

The final recommendation system uses the model with the strongest balance of predictive performance and interpretability.

If the random forest performs better, it is used for predicted win probability. The LASSO model remains useful for interpretation because it shows which serve and context variables are most strongly associated with point outcomes.

In [ ]:
import joblib
os.makedirs("../models", exist_ok=True)
rf_model.fit(X, y)
joblib.dump(rf_model, "../models/serve_win_probability_model.pkl")
joblib.dump(model_features, "../models/model_features.pkl")
print("Model saved to ../models/serve_win_probability_model.pkl")
joblib.dump(gb_model.fit(X, y), "../models/gradient_boosting_model.pkl")
print("Gradient Boosting model saved to ../models/gradient_boosting_model.pkl")

## Additional Validation Metrics
Evaluate ranking quality and calibration-sensitive error for better model selection.


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import brier_score_loss, average_precision_score

model_oof_probs = {}
for name, pipe in {'lasso': lasso_model, 'rf': rf_model, 'gb': gb_model}.items():
    model_oof_probs[name] = cross_val_predict(pipe, X, y, cv=cv, groups=groups, method='predict_proba')[:, 1]

extra_metrics = pd.DataFrame({
    'model': ['LASSO Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'average_precision': [
        average_precision_score(y, model_oof_probs['lasso']),
        average_precision_score(y, model_oof_probs['rf']),
        average_precision_score(y, model_oof_probs['gb'])
    ],
    'brier_loss': [
        brier_score_loss(y, model_oof_probs['lasso']),
        brier_score_loss(y, model_oof_probs['rf']),
        brier_score_loss(y, model_oof_probs['gb'])
    ]
}).sort_values('average_precision', ascending=False)
extra_metrics
